# Synthetic DEM Creation

This notebook:
- Reads and summarizes all available DEMs.
- Splits them into 64x64 and 128x128 groups.
- Selects 20 DEMs (64x64) stratified by relief, and keeps all 10 DEMs (128x128).
- Generates sloped versions (1% and 2%) for both groups.
- Saves everything into a new Synthetic_DEMs folder.

### Imports

In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import sys 

sys.path.insert(0,r"C:\Users\Prof. KN Jha\Desktop\SFINCS_mSWE-GNN_v3")
from scripts.sfincs_dataset_creation.dem import list_dem_files, read_dem_file, summarize_dem

### Paths

In [2]:
PROJECT_ROOT = Path(r"C:\Users\Prof. KN Jha\Desktop\SFINCS_mSWE-GNN_v3")
DEM_FOLDER = PROJECT_ROOT / "data" / "dem" / "Raw_DEMs" / "DEM"  # Folder containing DEM_*.txt files.
SYNTHETIC_FOLDER = PROJECT_ROOT / "data" / "Synthetic_DEMs"
SYNTHETIC_FOLDER.mkdir(parents=True, exist_ok=True)

print("DEM folder:", DEM_FOLDER)
print("DEM Folder exists:", DEM_FOLDER.exists())
print("Synthetic folder:", SYNTHETIC_FOLDER)

DEM folder: C:\Users\Prof. KN Jha\Desktop\SFINCS_mSWE-GNN_v3\data\dem\Raw_DEMs\DEM
DEM Folder exists: True
Synthetic folder: C:\Users\Prof. KN Jha\Desktop\SFINCS_mSWE-GNN_v3\data\Synthetic_DEMs


## Summarize every DEM

In [3]:
dem_files = list_dem_files(DEM_FOLDER)

summary_rows = []
for dem in dem_files:
    dem_path = DEM_FOLDER / dem
    stats = summarize_dem(dem_path)
    summary_rows.append(stats)

dem_stats = pd.DataFrame(summary_rows)
print("Total DEMs found:", len(dem_stats))
dem_stats.head()

Total DEMs found: 131


,dem_name,nx,ny,min_elevation_m,max_elevation_m,mean_elevation_m,std_elevation_m,relief_m
0,DEM_1.txt,64,64,-3.44297,3.03812,0.006227,1.144541,6.48109
1,DEM_10.txt,64,64,-1.84128,2.09117,0.004841,0.656046,3.93245
2,DEM_10000.txt,64,64,-2.82632,2.93463,0.006893,0.998018,5.76095
3,DEM_10001.txt,64,64,-2.98288,3.21417,0.065048,1.024340,6.19705
4,DEM_10002.txt,64,64,-1.05224,0.89629,-0.002550,0.315036,1.94853


### Split DEMs by grid size

In [4]:
is_64x64 = (dem_stats["nx"] == 64) & (dem_stats["ny"] == 64)
is_128x128 = (dem_stats["nx"] == 128) & (dem_stats["ny"] == 128)
dems_64x64 = dem_stats[is_64x64]
dems_128x128 = dem_stats[is_128x128].reset_index(drop=True) #reset index for 128x128 DEMs to avoid confusion with original indices
print("64 x 64 DEMs:", len(dems_64x64))
print("128 x 128 DEMs:", len(dems_128x128))

64 x 64 DEMs: 121
128 x 128 DEMs: 10


In [10]:
df = pd.DataFrame(dems_128x128)
df["relief_m"].mean()

np.float64(3.7765660166740416)

### Select 20 DEMs (64 x 64) stratified by relief

In [11]:
sorted_dems_64x64 = dems_64x64.sort_values("relief_m")
total_dems = len(sorted_dems_64x64)
num_groups = 4
group_size = total_dems // num_groups
group_1 = sorted_dems_64x64.iloc[0:group_size]
group_2 = sorted_dems_64x64.iloc[group_size:2*group_size]
group_3 = sorted_dems_64x64.iloc[2*group_size:3*group_size]
group_4 = sorted_dems_64x64.iloc[3*group_size:4*group_size]

num_per_group = 5
picked_1 = group_1.sample(n=num_per_group, random_state=42)
picked_2 = group_2.sample(n=num_per_group, random_state=42)
picked_3 = group_3.sample(n=num_per_group, random_state=42)
picked_4 = group_4.sample(n=num_per_group, random_state=42)

selected_64x64 = pd.concat([picked_1, picked_2, picked_3, picked_4]).reset_index(drop=True)
print("Selected 64 x 64 DEMs:", len(selected_64x64))
selected_64x64

Selected 64 x 64 DEMs: 20


,dem_name,nx,ny,min_elevation_m,max_elevation_m,mean_elevation_m,std_elevation_m,relief_m
0,DEM_41.txt,64,64,-1.32355,1.15175,-0.041646,0.409238,2.47530
1,DEM_37.txt,64,64,-0.85846,0.97150,-0.000567,0.281631,1.82996
2,DEM_62.txt,64,64,-1.24819,0.99870,0.009214,0.351902,2.24689
3,DEM_10002.txt,64,64,-1.05224,0.89629,-0.002550,0.315036,1.94853
4,DEM_511.txt,64,64,-0.80176,0.86501,-0.014883,0.293259,1.66677
5,DEM_38.txt,64,64,-2.24994,1.84869,-0.011790,0.642514,4.09863
6,DEM_47.txt,64,64,-1.85962,1.59113,-0.022967,0.601166,3.45075
7,DEM_3.txt,64,64,-1.86925,2.15570,0.011565,0.701113,4.02495
8,DEM_512.txt,64,64,-1.66006,1.90800,0.017959,0.572564,3.56806
9,DEM_30.txt,64,64,-1.73903,1.37377,0.004370,0.504824,3.11280


### Keep All 10 DEMs (128 x 128) for testing

In [12]:
selected_128x128 = dems_128x128.copy()
print("Selected 128 x 128 DEMs:", len(selected_128x128))
selected_128x128

Selected 128 x 128 DEMs: 10


,dem_name,nx,ny,min_elevation_m,max_elevation_m,mean_elevation_m,std_elevation_m,relief_m
0,DEM_15001.txt,128,128,-3.16006,3.34007,-0.010749,1.012836,6.50013
1,DEM_15002.txt,128,128,-1.97743,2.01657,-0.006554,0.634822,3.99400
2,DEM_15003.txt,128,128,-1.32346,1.43866,0.004377,0.441611,2.76212
3,DEM_15004.txt,128,128,-1.99583,2.88970,0.010101,0.730458,4.88553
4,DEM_15005.txt,128,128,-0.81477,0.78471,0.003065,0.267369,1.59948
5,DEM_15006.txt,128,128,-0.96495,1.15185,0.002979,0.314679,2.11680
6,DEM_15007.txt,128,128,-3.19266,2.94137,-0.004082,0.880613,6.13403
7,DEM_15008.txt,128,128,-1.01344,0.95909,-0.005861,0.304584,1.97253
8,DEM_15009.txt,128,128,-2.49618,2.24088,0.025500,0.770201,4.73706
9,DEM_15010.txt,128,128,-1.66052,1.40346,0.010117,0.429441,3.06398


### Save the flat DEM selections

In [13]:
selected_64x64.to_csv(SYNTHETIC_FOLDER / "selected_flat_64x64.csv", index=False)
selected_128x128.to_csv(SYNTHETIC_FOLDER / "selected_flat_128x128.csv", index=False)
print("Saved flat DEM selections")

Saved flat DEM selections


### Define the slope-adding function

In [14]:
def add_slope(dem_grid, dx, total_relief_m, direction_deg):
    """Tilt a DEM so the total elevation drop across the domain equals total_relief_m,
    regardless of grid size or dx."""
    ny, nx = dem_grid.shape
    i_positions = np.arange(ny)
    j_positions = np.arange(nx)
    j_mesh, i_mesh = np.meshgrid(j_positions, i_positions)

    direction_rad = np.deg2rad(direction_deg)

    raw_ramp = i_mesh * dx * np.cos(direction_rad) + j_mesh * dx * np.sin(direction_rad)
    normalized_ramp = (raw_ramp - raw_ramp.min()) / (raw_ramp.max() - raw_ramp.min()) #Normalizes the ramp to a 0-1 range
    """After normalization, the ramp goes from 0 to 1 across the DEM. We want it to go from -0.5 to +0.5, so we subtract 0.5. Then we multiply by total_relief_m to scale it to the desired elevation drop."""
    slope_array = (normalized_ramp - 0.5) * total_relief_m  #Centers the slope around 0, so the middle of the DEM is at the original elevation, and the edges drop by half the total relief.

    dem_sloped = dem_grid + slope_array
    return dem_sloped



def write_dem_txt(dem_path, dem_grid, x_coords, y_coords):
    X, Y = np.meshgrid(x_coords, y_coords) #Using meshgrid to create 2D arrays of x and y coordinates

    # Step 2: flatten each 2D grid into one long list of numbers (row by row).
    x_flat = X.ravel()          # every cell's x-coordinate, one after another
    y_flat = Y.ravel()          # every cell's y-coordinate, one after another
    z_flat = dem_grid.ravel()   # every cell's elevation, one after another

    xyz = np.column_stack([x_flat, y_flat, z_flat]) # Step 3: place the three lists side by side, so each row becomes one (x, y, z) point.
    np.savetxt(dem_path, xyz, fmt=("%1.1f", "%1.1f", "%1.5f")) # Step 4: write that table to a text file, with x and y rounded to 1 decimal and elevation to 5 decimals for precision.


### Sloped DEM generation

In [15]:
sloped_dem = []
random.seed(42)

gentle_64x64 = selected_64x64.iloc[0:5]
steep_64x64 = selected_64x64.iloc[5:10]

for row_index, row in gentle_64x64.iterrows():
    dem_grid, x_coords, y_coords = read_dem_file(DEM_FOLDER / row["dem_name"])
    dx = float(x_coords[1] - x_coords[0])
    direction_deg = random.uniform(0, 360)
    dem_sloped = add_slope(dem_grid, dx, total_relief_m=5.0, direction_deg=direction_deg)

    output_name = f"{Path(row['dem_name']).stem}_sloped_5m_{int(direction_deg)}deg.txt"
    write_dem_txt(SYNTHETIC_FOLDER / output_name, dem_sloped, x_coords, y_coords)

    sloped_dem.append({
        "source_dem": row["dem_name"],
        "output_dem": output_name,
        "grid": "64x64",
        "category": "moderate",
        "target_relief_m": 5.0,
        "direction_deg": round(direction_deg, 2),
        "relief_m": float(dem_sloped.max() - dem_sloped.min()),
        "dataset_role": "train",
    })

for row_index, row in steep_64x64.iterrows():
    dem_grid, x_coords, y_coords = read_dem_file(DEM_FOLDER / row["dem_name"])
    dx = float(x_coords[1] - x_coords[0])
    direction_deg = random.uniform(0, 360)
    dem_sloped = add_slope(dem_grid, dx, total_relief_m=10.0, direction_deg=direction_deg)

    output_name = f"{Path(row['dem_name']).stem}_sloped_10m_{int(direction_deg)}deg.txt"
    write_dem_txt(SYNTHETIC_FOLDER / output_name, dem_sloped, x_coords, y_coords)

    sloped_dem.append({
        "source_dem": row["dem_name"],
        "output_dem": output_name,
        "grid": "64x64",
        "category": "steep",
        "target_relief_m": 10.0,
        "direction_deg": round(direction_deg, 2),
        "relief_m": float(dem_sloped.max() - dem_sloped.min()),
        "dataset_role": "train",
    })

In [16]:
gentle_128x128 = selected_128x128.iloc[0:3]
steep_128x128 = selected_128x128.iloc[3:6]

for row_index, row in gentle_128x128.iterrows():
    dem_grid, x_coords, y_coords = read_dem_file(DEM_FOLDER / row["dem_name"])
    dx = float(x_coords[1] - x_coords[0])
    direction_deg = random.uniform(0, 360)
    dem_sloped = add_slope(dem_grid, dx, total_relief_m=5.0, direction_deg=direction_deg)

    output_name = f"{Path(row['dem_name']).stem}_sloped_5m_{int(direction_deg)}deg.txt"
    write_dem_txt(SYNTHETIC_FOLDER / output_name, dem_sloped, x_coords, y_coords)

    sloped_dem.append({
        "source_dem": row["dem_name"],
        "output_dem": output_name,
        "grid": "128x128",
        "category": "moderate",
        "target_relief_m": 5.0,
        "direction_deg": round(direction_deg, 2),
        "relief_m": float(dem_sloped.max() - dem_sloped.min()),
        "dataset_role": "test",
    })

for row_index, row in steep_128x128.iterrows():
    dem_grid, x_coords, y_coords = read_dem_file(DEM_FOLDER / row["dem_name"])
    dx = float(x_coords[1] - x_coords[0])
    direction_deg = random.uniform(0, 360)
    dem_sloped = add_slope(dem_grid, dx, total_relief_m=10.0, direction_deg=direction_deg)

    output_name = f"{Path(row['dem_name']).stem}_sloped_10m_{int(direction_deg)}deg.txt"
    write_dem_txt(SYNTHETIC_FOLDER / output_name, dem_sloped, x_coords, y_coords)

    sloped_dem.append({
        "source_dem": row["dem_name"],
        "output_dem": output_name,
        "grid": "128x128",
        "category": "steep",
        "target_relief_m": 10.0,
        "direction_deg": round(direction_deg, 2),
        "relief_m": float(dem_sloped.max() - dem_sloped.min()),
        "dataset_role": "test",
    })

sloped_dem_summary = pd.DataFrame(sloped_dem)
print(sloped_dem_summary["dataset_role"].value_counts())
print(sloped_dem_summary["grid"].value_counts())
sloped_dem_summary

dataset_role
train    10
test      6
Name: count, dtype: int64
grid
64x64      10
128x128     6
Name: count, dtype: int64


,source_dem,output_dem,grid,category,target_relief_m,direction_deg,relief_m,dataset_role
0,DEM_41.txt,DEM_41_sloped_5m_230deg.txt,64x64,moderate,5.0,230.19,5.410504,train
1,DEM_37.txt,DEM_37_sloped_5m_9deg.txt,64x64,moderate,5.0,9.00,5.453560,train
2,DEM_62.txt,DEM_62_sloped_5m_99deg.txt,64x64,moderate,5.0,99.01,6.420192,train
3,DEM_10002.txt,DEM_10002_sloped_5m_80deg.txt,64x64,moderate,5.0,80.36,5.932682,train
4,DEM_511.txt,DEM_511_sloped_5m_265deg.txt,64x64,moderate,5.0,265.13,5.950066,train
5,DEM_38.txt,DEM_38_sloped_10m_243deg.txt,64x64,steep,10.0,243.61,11.352041,train
6,DEM_47.txt,DEM_47_sloped_10m_321deg.txt,64x64,steep,10.0,321.18,10.445281,train
7,DEM_3.txt,DEM_3_sloped_10m_31deg.txt,64x64,steep,10.0,31.30,10.832480,train
8,DEM_512.txt,DEM_512_sloped_10m_151deg.txt,64x64,steep,10.0,151.89,11.214137,train
9,DEM_30.txt,DEM_30_sloped_10m_10deg.txt,64x64,steep,10.0,10.73,10.729074,train


In [17]:
sloped_dem_summary = pd.DataFrame(sloped_dem)
print(sloped_dem_summary["dataset_role"].value_counts())
print(sloped_dem_summary["grid"].value_counts())
sloped_dem_summary

dataset_role
train    10
test      6
Name: count, dtype: int64
grid
64x64      10
128x128     6
Name: count, dtype: int64


,source_dem,output_dem,grid,category,target_relief_m,direction_deg,relief_m,dataset_role
0,DEM_41.txt,DEM_41_sloped_5m_230deg.txt,64x64,moderate,5.0,230.19,5.410504,train
1,DEM_37.txt,DEM_37_sloped_5m_9deg.txt,64x64,moderate,5.0,9.00,5.453560,train
2,DEM_62.txt,DEM_62_sloped_5m_99deg.txt,64x64,moderate,5.0,99.01,6.420192,train
3,DEM_10002.txt,DEM_10002_sloped_5m_80deg.txt,64x64,moderate,5.0,80.36,5.932682,train
4,DEM_511.txt,DEM_511_sloped_5m_265deg.txt,64x64,moderate,5.0,265.13,5.950066,train
5,DEM_38.txt,DEM_38_sloped_10m_243deg.txt,64x64,steep,10.0,243.61,11.352041,train
6,DEM_47.txt,DEM_47_sloped_10m_321deg.txt,64x64,steep,10.0,321.18,10.445281,train
7,DEM_3.txt,DEM_3_sloped_10m_31deg.txt,64x64,steep,10.0,31.30,10.832480,train
8,DEM_512.txt,DEM_512_sloped_10m_151deg.txt,64x64,steep,10.0,151.89,11.214137,train
9,DEM_30.txt,DEM_30_sloped_10m_10deg.txt,64x64,steep,10.0,10.73,10.729074,train


In [18]:
measured_relief = []

for idx, row in sloped_dem_summary.iterrows():
    dem_grid, xc, yc = read_dem_file(SYNTHETIC_FOLDER / row["output_dem"])
    dx = float(xc[1] - xc[0])

    actual_relief_m = float(dem_grid.max() - dem_grid.min())
    gy, gx = np.gradient(dem_grid, dx)
    local_grade_percent = 100 * np.sqrt(np.mean(gx)**2 + np.mean(gy)**2)

    print(f"{row['output_dem']}: target={row['target_relief_m']:.1f} m | "
          f"actual={actual_relief_m:.2f} m | local grade={local_grade_percent:.4f}%")
    measured_relief.append(actual_relief_m)

sloped_dem_summary["measured_relief_m"] = measured_relief

DEM_41_sloped_5m_230deg.txt: target=5.0 m | actual=5.41 m | local grade=0.0521%
DEM_37_sloped_5m_9deg.txt: target=5.0 m | actual=5.45 m | local grade=0.0690%
DEM_62_sloped_5m_99deg.txt: target=5.0 m | actual=6.42 m | local grade=0.0702%
DEM_10002_sloped_5m_80deg.txt: target=5.0 m | actual=5.93 m | local grade=0.0685%
DEM_511_sloped_5m_265deg.txt: target=5.0 m | actual=5.95 m | local grade=0.0776%
DEM_38_sloped_10m_243deg.txt: target=10.0 m | actual=11.35 m | local grade=0.1202%
DEM_47_sloped_10m_321deg.txt: target=10.0 m | actual=10.45 m | local grade=0.1124%
DEM_3_sloped_10m_31deg.txt: target=10.0 m | actual=10.83 m | local grade=0.1130%
DEM_512_sloped_10m_151deg.txt: target=10.0 m | actual=11.21 m | local grade=0.1160%
DEM_30_sloped_10m_10deg.txt: target=10.0 m | actual=10.73 m | local grade=0.1352%
DEM_15001_sloped_5m_78deg.txt: target=5.0 m | actual=9.83 m | local grade=0.0317%
DEM_15002_sloped_5m_181deg.txt: target=5.0 m | actual=8.15 m | local grade=0.0380%
DEM_15003_sloped_5m_9d

In [19]:
sloped_dem_summary.to_csv(SYNTHETIC_FOLDER / "sloped_dem_summary.csv", index=False) #index =False prevents pandas from writing row indices to the CSV file